In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

textfile = open("12383.txt", "r") 
text = textfile.read()
print(f"**** The training text /n{text[:200]} /n *************8")
chars = sorted(list(set(text)))
stoi = {ch :i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encoded = [stoi[c] for c in text]

class CharDataset(Dataset):
    def __init__(self,data,context_size =5):
        self.data = data
        self.context_size=context_size
        
    def __len__(self):
        return len(self.data)-self.context_size
    def __getitem__(self,idx):
        x = torch.tensor(self.data[idx:idx+self.context_size])
        y = torch.tensor(self.data[idx + self.context_size])
        return x,y

dataset = CharDataset(encoded,context_size=5)
dataloader = DataLoader(dataset,batch_size=4,shuffle=True)



**** The training text /n  "She was a Phantom of delight"

  "I wandered lonely as a cloud"

  The Affliction of Margaret--

  The Forsaken

  Repentance

  Address to my Infant Daughter, Dora

  The Kitten and Falling Leaves /n *************8


In [8]:
print(f"**** The dataset length is {len(dataset)} ****")

**** The dataset length is 812879 ****


In [9]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)                      # (batch, seq, embed)
        out, hidden = self.lstm(x, hidden)     # (batch, seq, hidden)
        out = self.fc(out[:, -1, :])           # last time step
        return out, hidden


In [10]:
import torch.nn as nn
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embed_dim,hidden_size,num_layers,dropout):
        super(LSTMModel,self).__init__()
        self.embed = nn.Embedding(vocab_size,embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc=nn.Linear(hidden_size,vocab_size)
    def forward(self,x,hidden=None):
        x = self.embed(x)
        out,hidden = self.lstm(x,hidden)
        out = self.fc(out[:,-1,:])
        return out,hidden
        
vocab_size = len(stoi)
embed_dim = 32         
hidden_size = 64       
num_layers = 2        
dropout = 0.1


In [11]:
model = LSTMModel(vocab_size, embed_dim, hidden_size, num_layers, dropout)
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import torch.backends.cudnn
torch.backends.cudnn.benchmark = True

Using device: cpu


In [12]:
epochs = 50

for epoch in range(epochs):
    total_loss = 0
    for xb, yb in dataloader:
        out, _ = model(xb)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")



Epoch 1: Loss = 401268.3227
Epoch 2: Loss = 400211.7765


KeyboardInterrupt: 

In [ ]:
def predict_next_token(model, prompt, num_preds=10):
    model.eval()
    input_seq = torch.tensor([stoi[c] for c in prompt[-5:]]).unsqueeze(0)
    preds = []

    hidden = None
    for _ in range(num_preds):
        out, hidden = model(input_seq, hidden)
        prob = torch.softmax(out, dim=1)
        next_id = torch.argmax(prob, dim=1).item()
        preds.append(itos[next_id])
        input_seq = torch.cat([input_seq[:, 1:], torch.tensor([[next_id]])], dim=1)
    return ''.join(preds)

# Try prediction
print(predict_next_token(model, "hello"))
